# NYC Permitted Event Addresses

**Source:** `data/NYC_Permitted_Event_Information_-_Historical_20260927.csv`  
**Scope:** events starting in 2016 or 2017 (January 1, 2016 through December 31, 2017).  
**Goal:** list and export unique event locations from this two-year period.


In [28]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", None)

CSV_NAME = "NYC_Permitted_Event_Information_-_Historical_20260927.csv"
# Optional override for kernels running outside the repository.
# Set NYC_EVENTS_CSV to the full CSV path before running this cell.
configured_path = os.environ.get("NYC_EVENTS_CSV")
if configured_path:
    candidates = [Path(configured_path).expanduser()]
else:
    roots = [
        Path.cwd(), *Path.cwd().parents,
        Path("/app"),  # docker-compose mounts the repository here
        Path.home() / "code" / "mle-citibike-demand-forecasting",
    ]
    candidates = list(dict.fromkeys(root / "data" / CSV_NAME for root in roots))

DATA_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    checked = "\n".join(f"  {p}" for p in candidates)
    raise FileNotFoundError(
        f"NYC event CSV is not visible to this kernel. Working directory: {Path.cwd()}\n"
        f"Checked:\n{checked}\n"
        "Set os.environ['NYC_EVENTS_CSV'] to the CSV's full path in this kernel's filesystem."
    )
ROOT = DATA_PATH.parent.parent

events = pd.read_csv(DATA_PATH)
rows_loaded = len(events)
event_start_dates = pd.to_datetime(
    events["Start Date/Time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
)
# Keep only events whose start year is 2016 or 2017.
FILTER_YEARS = [2016, 2017]
in_period = event_start_dates.dt.year.isin(FILTER_YEARS)
events = events.loc[in_period].copy()
print(f"Events starting in 2016–2017: {len(events):,} of {rows_loaded:,} rows")
print(f"Rows excluded for invalid/missing start dates: {event_start_dates.isna().sum():,}")
print(f"Source: {DATA_PATH}")
print(f"Shape: {len(events):,} rows x {len(events.columns)} columns")
print(f"Columns: {list(events.columns)}")


/var/folders/5c/ppdgvcnj63x80zyn8b9w2z_h0000gn/T/ipykernel_73895/2061348915.py:33: DtypeWarning: Columns (0: Event Street Side, 1: Street Closure Type) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv(DATA_PATH)


Events starting in 2016–2017: 425,425 of 2,906,868 rows
Rows excluded for invalid/missing start dates: 0
Source: /Users/minsan/code/mle-citibike-demand-forecasting/data/NYC_Permitted_Event_Information_-_Historical_20260927.csv
Shape: 425,425 rows x 13 columns
Columns: ['Event ID', 'Event Name', 'Start Date/Time', 'End Date/Time', 'Event Agency', 'Event Type', 'Event Borough', 'Event Location', 'Event Street Side', 'Street Closure Type', 'Community Board', 'Police Precinct', 'CEMSID']


In [29]:
# Preview the first five events after filtering to 2016 and 2017.
display(events.head())


,Event ID,Event Name,Start Date/Time,End Date/Time,Event Agency,Event Type,Event Borough,Event Location,Event Street Side,Street Closure Type,Community Board,Police Precinct,CEMSID
30911,192645,4th Annual Bronx Recreation WinterFest 2015,01/31/2016 11:00:00 AM,01/31/2016 04:00:00 PM,Parks Department,Special Event,Bronx,Williamsbridge Oval: Scenic Tables,NaN,NaN,"7,","52,","1377,"
42066,237036,Picnic,08/13/2016 12:00:00 PM,08/13/2016 07:30:00 PM,Parks Department,Special Event,Brooklyn,"Dyker Beach Park: South Lawn ,Dyker Beach Park: Dyker Bay 8-Softball-04",NaN,NaN,"10,","68,","4468, 7787,"
42306,237998,ADS Color Run,06/04/2016 10:00:00 AM,06/04/2016 01:00:00 PM,Parks Department,Special Event,Bronx,St. Mary's Park: Track-01,NaN,NaN,"01,","40,","7017,"
43844,245816,Lacrosse,03/12/2016 09:00:00 AM,03/12/2016 04:00:00 PM,Parks Department,Sport - Youth,Manhattan,"Randall's Island Park: East River Fields-Soccer-82 ,Randall's Island Park: East River Fields-Soc...",NaN,NaN,"11,","25,","5753, 5756, 5757, 5758, 5745, 5750, 5747, 5749, 5755, 5754, 5748,"
44350,251353,Celebration,05/28/2016 02:00:00 PM,05/28/2016 07:00:00 PM,Parks Department,Special Event,Manhattan,"Fort Tryon Park: Linden Terrace ,Fort Tryon Park: Heather Garden ,Fort Tryon Park: Billings Lawn",NaN,NaN,"12,","34,","4007, 4005, 3994,"


In [30]:
print(f"Total events: {len(events):,}")
print(f"Events with a location: {events['Event Location'].notna().sum():,}")
print(f"Events without a location: {events['Event Location'].isna().sum():,}")
print("Events by start year:")
print(event_start_dates.loc[in_period].dt.year.value_counts().sort_index())


Total events: 425,425
Events with a location: 425,335
Events without a location: 90
Events by start year:
Start Date/Time
2016    208532
2017    216893
Name: count, dtype: int64


In [31]:
# Counts by street closure type for events starting in 2016–2017.
street_closure_types_counts = (
    events["Street Closure Type"]
    .value_counts()
    .rename_axis("Street Closure Type")
    .reset_index(name="Count")
)
unique_street_closure_types = sorted(street_closure_types_counts["Street Closure Type"].tolist())
print(f"Unique street closure type values: {len(unique_street_closure_types)}")
display(street_closure_types_counts)
print(f"Missing street closure type: {events['Street Closure Type'].isna().sum():,}")


Unique street closure type values: 7


,Street Closure Type,Count
0,Full Street Closure,8321
1,Curb Lane Only,2814
2,Partial Sidewalk Closure,2800
3,Sidewalk and Curb Lane Closure,2320
4,Pedestrian Plaza,1126
5,Sidewalk and Street Closure,616
6,Full Sidewalk Closure,507


Missing street closure type: 406,921


In [32]:
# Counts by event type for events starting in 2016–2017.
event_types_counts = (
    events["Event Type"]
    .value_counts()
    .rename_axis("Event Type")
    .reset_index(name="Count")
)
unique_event_types = sorted(event_types_counts["Event Type"].tolist())
print(f"Unique event type values: {len(unique_event_types)}")
display(event_types_counts)
print(f"Missing event type: {events['Event Type'].isna().sum():,}")


Unique event type values: 33


,Event Type,Count
0,Sport - Youth,207434
1,Sport - Adult,130827
2,Special Event,68212
3,Block Party,4084
4,Farmers Market,3291
5,Production Event,2180
6,Parade,1675
7,Street Event,1524
8,Plaza Event,952
9,Miscellaneous,889


Missing event type: 0


In [33]:
unique_locations = events["Event Location"].dropna().unique()
print(f"Number of unique event locations: {len(unique_locations):,}")
print()
for i, loc in enumerate(unique_locations, 1):
    print(f"{i}. {loc}")

Number of unique event locations: 18,197

1. Williamsbridge Oval: Scenic Tables
2. Dyker Beach Park: South Lawn ,Dyker Beach Park: Dyker Bay 8-Softball-04
3. St. Mary's Park: Track-01
4. Randall's Island Park: East River Fields-Soccer-82 ,Randall's Island Park: East River Fields-Soccer-85 ,Randall's Island Park: Sunken Garden Fields-Soccer-90 ,Randall's Island Park: Sunken Garden Fields-Soccer-91 ,Randall's Island Park: Wards Meadow Fields-Soccer-70 ,Randall's Island Park: Wards Meadow Fields-Soccer-75 ,Randall's Island Park: Wards Meadow Fields-Soccer-72 ,Randall's Island Park: Wards Meadow Fields-Soccer-74 ,Randall's Island Park: East River Fields-Soccer-84 ,Randall's Island Park: East River Fields-Soccer-83 ,Randall's Island Park: Wards Meadow Fields-Soccer-73
5. Fort Tryon Park: Linden Terrace ,Fort Tryon Park: Heather Garden ,Fort Tryon Park: Billings Lawn
6. Riverside Park: 72nd-95th St Riverside Dr-RSP ,Riverside Park: Promenade 100th-110th Street-RSP
7. Herbert Von King Park: L

In [34]:
location_counts = events["Event Location"].dropna().value_counts().reset_index()
location_counts.columns = ["Event Location", "Count"]
print(f"Top 30 most common event locations:")
display(location_counts.head(30))

Top 30 most common event locations:


,Event Location,Count
0,James J Walker Park: Soccer-01,2227
1,Bushwick Inlet Park: Soccer-01,2170
2,McCarren Park: Soccer-01,1712
3,Sara D. Roosevelt Park: Nike Field-Soccer-02,1622
4,East River Park: Grand Street Mini Field-Soccer-02,1535
5,Columbus Park: Soccer-01,1503
6,College Point Fields: Soccer-01,1451
7,"Sunset Park: Soccer-01A ,Sunset Park: Soccer-01B",1370
8,Chelsea Park: Soccer-01,1363
9,Sara D. Roosevelt Park: Lion's Gate-Soccer-03,1354


In [35]:
location_counts = events["Event Location"].dropna().value_counts().reset_index()
location_counts.columns = ["Event Location", "Count"]
location_counts.to_csv(ROOT / "data" / "unique_event_locations.csv", index=False)
print(f"Saved {len(location_counts)} unique locations to data/unique_event_locations.csv")

Saved 18197 unique locations to data/unique_event_locations.csv


## Relevant event types: before and after

Compare event record counts and unique location strings within 2016–2017. Locations can describe parks, street segments, or multiple addresses. Trip counts require a separate spatial and time-window match to trip data.


In [36]:
relevant_event_types = [
    "Athletic Race / Tour",
    "Parade",
    "Street Festival",
    "Single Block Festival",
    "Block Party",
    "Street Event",
    "Play Streets",
    "Farmers Market",
]

# `events` already contains only events starting in 2016 or 2017.
filtered_events = events.loc[
    events["Event Type"].str.strip().isin(relevant_event_types)
].copy()

comparison = pd.DataFrame([
    {
        "Dataset": "Before event-type filter (2016–2017)",
        "Event records": len(events),
        "Unique addresses": events["Event Location"].nunique(),
    },
    {
        "Dataset": "After event-type filter (2016–2017)",
        "Event records": len(filtered_events),
        "Unique addresses": filtered_events["Event Location"].nunique(),
    },
]).set_index("Dataset")
display(comparison)
print(f"Event records removed: {len(events) - len(filtered_events):,}")
print("Unique addresses count distinct non-null Event Location strings.")


,Event records,Unique addresses
Dataset,,
Before event-type filter (2016–2017),425425,18197
After event-type filter (2016–2017),12372,6667


Event records removed: 413,053
Unique addresses count distinct non-null Event Location strings.


In [37]:
# Counts for every requested type, including zero matches.
filtered_event_type_counts = (
    filtered_events["Event Type"].str.strip().value_counts()
    .reindex(relevant_event_types, fill_value=0)
    .rename_axis("Event Type").reset_index(name="Count")
)
display(filtered_event_type_counts)

# Available for geocoding or further analysis without changing the earlier export.
filtered_location_counts = (
    filtered_events["Event Location"].value_counts()
    .rename_axis("Event Location").reset_index(name="Count")
)
display(filtered_events.head())


,Event Type,Count
0,Athletic Race / Tour,219
1,Parade,1675
2,Street Festival,426
3,Single Block Festival,577
4,Block Party,4084
5,Street Event,1524
6,Play Streets,576
7,Farmers Market,3291


,Event ID,Event Name,Start Date/Time,End Date/Time,Event Agency,Event Type,Event Borough,Event Location,Event Street Side,Street Closure Type,Community Board,Police Precinct,CEMSID
224650,183663,Mexican Day Parade,09/18/2016 12:00:00 PM,09/18/2016 04:00:00 PM,Police Department,Parade,Manhattan,"EAST 37 STREET between FIFTH AVENUE and PARK AVENUE, EAST 38 STREET between FIFTH AVENUE an...",Full,Full Street Closure,"5, 6,","13, 14, 17,",NaN
227319,209482,Block party,06/04/2016 01:00:00 PM,06/04/2016 07:00:00 PM,Street Activity Permit Office,Block Party,Brooklyn,5 STREET between 6 AVENUE and 7 AVENUE,Both,Full Street Closure,"6,","78,",NaN
227565,211174,"Dominican Day Parade, Inc.",08/14/2016 12:00:00 PM,08/14/2016 04:00:00 PM,Police Department,Parade,Manhattan,"SIXTH AVENUE between WEST 37 STREET and WEST 38 STREET, SIXTH AVENUE between WEST 36 STRE...",Both,Full Street Closure,"5,","14, 18,",NaN
228664,220767,5 de Mayo Fiesta NY,05/01/2016 10:00:00 AM,05/01/2016 06:00:00 PM,Street Activity Permit Office,Single Block Festival,Manhattan,EAST 116 STREET between 3 AVENUE and 2 AVENUE,Full,Full Street Closure,"11,","25,",NaN
228940,223277,Haitian Flag Day Parade,05/22/2016 11:00:00 AM,05/22/2016 07:00:00 PM,Police Department,Parade,Brooklyn,"NOSTRAND AVENUE between STERLING STREET and GLENWOOD ROAD, NOSTRAND AVENUE between STERLING STR...",Full,Full Street Closure,"14, 17, 9,","67, 70, 71,",NaN
